# Course Skill Extraction Task

**Research:** An End-to-End Study on Prompt Engineering Techniques in Generative AI  
for Automated Course Understanding, Skill Extraction, and Personalized Learning Insights

**Setup and Execution Guide**

Step 1: Environment Setup

Install all required libraries and dependencies.

Step 2: Pipeline Configuration and Execution

This step covers dataset loading, prompt setup, model configuration, evaluation and exporting results.

CONFIG section at the top to be updated as per the need of experiments.

In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================

import subprocess
import sys

SEPARATOR_WIDTH: int = 55

LIBRARY_INSTALLS = [
    ("openai>=1.30.0",),
    ("pandas>=2.0.0", "numpy>=1.24.0", "scipy>=1.10.0"),
    ("matplotlib>=3.7.0",),
]

BERT_INSTALLS = [
    ("torch",),
    ("transformers==4.35.2",),
    ("bert-score==0.3.13",),
]

VERSION_CHECK_LIBS: list = [
    ("openai",       "openai"),
    ("pandas",       "pandas"),
    ("numpy",        "numpy"),
    ("torch",        "torch"),
    ("transformers", "transformers"),
    ("bert_score",   "bert_score"),
    ("matplotlib",   "matplotlib"),
    ("scipy",        "scipy"),
]


def pip_install(*packages: str) -> None:
    """Install one or more pip packages quietly via subprocess."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )


def print_library_versions(lib_map: list) -> None:
    """Print installed version for each library in lib_map.

    Args:
        lib_map: List of (import_name, display_name) tuples.
    """
    print("=" * SEPARATOR_WIDTH)
    print("INSTALLED LIBRARY VERSIONS")
    print("=" * SEPARATOR_WIDTH)
    for import_name, display_name in lib_map:
        try:
            mod = __import__(import_name)
            version = getattr(mod, "__version__", "unknown")
            print(f"  {display_name:<18} {version}")
        except ImportError:
            print(f"  {display_name:<18} NOT INSTALLED")
    print("=" * SEPARATOR_WIDTH)


print("[1/4] Installing core packages...")
for pkg_group in LIBRARY_INSTALLS:
    pip_install(*pkg_group)

print("[2/4] Installing PyTorch...")
pip_install(*BERT_INSTALLS[0])

print("[3/4] Installing Transformers (pinned 4.35.2)...")
pip_install(*BERT_INSTALLS[1])

print("[4/4] Installing BERTScore (pinned 0.3.13)...")
pip_install(*BERT_INSTALLS[2])

print()
print_library_versions(VERSION_CHECK_LIBS)
print()
print("All packages installed successfully.")
print("If Colab prompts a runtime restart, click Restart.")
print("After restarting, run Cell 2 directly — do NOT re-run Cell 1.")

[1/4] Installing core packages...
[2/4] Installing PyTorch...
[3/4] Installing Transformers (pinned 4.35.2)...
[4/4] Installing BERTScore (pinned 0.3.13)...

INSTALLED LIBRARY VERSIONS
  openai             2.32.0
  pandas             2.2.2
  numpy              2.0.2
  torch              2.10.0+cpu
  transformers       4.35.2
  bert_score         0.3.12
  matplotlib         3.10.0
  scipy              1.16.3

All packages installed successfully.
If Colab prompts a runtime restart, click Restart.
After restarting, run Cell 2 directly — do NOT re-run Cell 1.


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
# =============================================================================
# CELL 2 — END-TO-END SKILL EXTRACTION EXPERIMENT
#
# Sections
# --------
#   A.  Imports & library version display
#   B.  *** TEMPLATE VARIABLES — edit here to create a new experiment ***
#   C.  Configuration
#   D.  Dataset load
#   E.  API key setup
#   F.  _build_prompt() for a new technique ***
#   G.  LLM caller
#   H.  Run experiment
#   I.  Skill parsing & evaluation metrics
#   J.  Full metrics table
#   K.  Final summary
#   L.  Save & download CSVs
# =============================================================================


# =============================================================================
# A.  IMPORTS & LIBRARY VERSION DISPLAY
# =============================================================================

import html
import getpass
import math
import os
import random
import re
import time
import warnings
from collections import Counter
from datetime import datetime, timezone
from typing import Dict, List, Optional, Set, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive
from openai import OpenAI

try:
    from google.colab import files
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

BERT_AVAILABLE: bool = False
BERT_WARNED: bool = False
try:
    from bert_score import score as bert_score_fn
    BERT_AVAILABLE = True
except ImportError:
    pass

matplotlib.rcParams["figure.dpi"] = 110

SEPARATOR_SM: int = 40
SEPARATOR_MD: int = 55
SEPARATOR_LG: int = 65
SEPARATOR_XL: int = 80
SEPARATOR_XXL: int = 90
TABLE_ROW_WIDTH: int = 105

LIBRARY_VERSION_MAP: List[Tuple[str, str]] = [
    ("openai",       "openai"),
    ("pandas",       "pandas"),
    ("numpy",        "numpy"),
    ("torch",        "torch"),
    ("transformers", "transformers"),
    ("bert_score",   "bert_score"),
    ("matplotlib",   "matplotlib"),
    ("scipy",        "scipy"),
]


def print_library_versions(lib_map: List[Tuple[str, str]]) -> None:
    """Print the active runtime version for each library.

    Args:
        lib_map: List of (import_name, display_name) tuples.
    """
    print("=" * SEPARATOR_MD)
    print("LIBRARY VERSIONS (active runtime)")
    print("=" * SEPARATOR_MD)
    for import_name, display_name in lib_map:
        try:
            mod = __import__(import_name)
            version = getattr(mod, "__version__", "unknown")
            print(f"  {display_name:<18} {version}")
        except ImportError:
            print(f"  {display_name:<18} NOT INSTALLED")
    print("=" * SEPARATOR_MD)
    print(f"  BERTScore available : {BERT_AVAILABLE}")
    print("=" * SEPARATOR_MD)


print_library_versions(LIBRARY_VERSION_MAP)


# =============================================================================
# B.  *** TEMPLATE VARIABLES ***
#
#  --------------
#  TECHNIQUE        : "zero_shot" | "few_shot" | "instruction" | "role_based"
#                     "chain_of_thought" | "self_consistency" | "react" | "tree_of_thought"
#  GROQ_SECRET_NAME : the Colab Secret name that holds the API key for this slot
# =============================================================================

TECHNIQUE: str = "self_consistency"
TASK_TYPE: str = "skill_extraction"
GROQ_SECRET_NAME: str = "GROQ_SELFCONSISTENCY"


# =============================================================================
# C.  CONFIGURATION
# =============================================================================

# Number of repeated runs  (1 = quick test | 3-5 = full research study)
NUM_RUNS: int = 1

# Temperature for skill extraction (low for factual, focused output)
TEMPERATURE: float = 0.2

TOP_P: float = 0.9

MODELS: List[str] = [
    "llama-3.1-8b-instant",
    "qwen/qwen3-32b",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
]

GROQ_BASE_URL: str = "https://api.groq.com/openai/v1"
MAX_TOKENS: int = 1024
DELAY_MIN: int = 7
DELAY_MAX: int = 12
N_ROWS: int = 25



def print_experiment_config() -> None:
    """Print a summary of the current experiment configuration."""
    total_calls = NUM_RUNS * len(MODELS) * N_ROWS
    avg_delay_min = total_calls * (DELAY_MIN + DELAY_MAX) / 2 / 60
    print("Experiment Configuration")
    print("-" * SEPARATOR_SM)
    print(f"  Technique          : {TECHNIQUE}")
    print(f"  Task type          : {TASK_TYPE}")
    print(f"  Temperature        : {TEMPERATURE}")
    print(f"  Runs               : {NUM_RUNS}")
    print(f"  Models             : {MODELS}")
    print(f"  Courses            : {N_ROWS}")
    print(f"  Total API calls    : {total_calls}")
    print(f"  Est. delay time    : ~{avg_delay_min:.1f} min")
    print(f"  top_p              : {TOP_P}")


print_experiment_config()


# =============================================================================
# D.  DATASET LOAD
# =============================================================================

COLUMN_RENAME_MAP: Dict[str, str] = {
    "title":        "course_title",
    "Organization": "course_organization",
    "Level":        "course_difficulty",
    "Description":  "description",
    "enrolled":     "course_students_enrolled",
    "rating":       "course_rating",
}

TEXT_COLUMNS: List[str] = [
    "course_title",
    "course_organization",
    "course_difficulty",
    "description",
]


def load_dataset(n_rows: int = N_ROWS) -> pd.DataFrame:
    """Load, rename, and clean the Coursera CSV dataset from Google Drive.
    Also loads and merges the gold skills CSV for evaluation.

    Args:
        n_rows: Maximum number of rows to load.

    Returns:
        Cleaned DataFrame with standardised columns and merged gold_skills.

    Raises:
        FileNotFoundError: If the dataset CSV is not found at the expected path.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("Dataset Load")
    print("-" * SEPARATOR_SM)
    print("  Loading Coursera dataset from Google Drive...")
    print("  Please wait...\n")

    drive.mount("/content/drive")

    dataset_path = "/content/drive/MyDrive/research/data/coursera_clean_sample.csv"
    gold_csv_path = "/content/drive/MyDrive/research/data/gold_skills_coursera.csv"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Dataset not found at '{dataset_path}'.\n"
            "Please ensure the file exists in your Google Drive."
        )

    print(f"  Found dataset at: {dataset_path}")
    print("  Cleaning and filtering rows...")

    df = pd.read_csv(dataset_path, encoding="latin-1")
    df = df.rename(columns=COLUMN_RENAME_MAP)
    df = df.dropna(subset=["course_title"]).head(n_rows).reset_index(drop=True)

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("N/A").astype(str)

    print(f"  Loaded {len(df)} courses successfully.")

    # Load gold skills
    if not os.path.exists(gold_csv_path):
        raise FileNotFoundError(
            f"Gold skills CSV not found at '{gold_csv_path}'.\n"
            "Expected file: gold_skills_coursera.csv in your Drive."
        )

    print(f"  Loading gold skills from: {gold_csv_path}")
    gold_df = pd.read_csv(gold_csv_path, encoding="latin-1")

    # Find the gold skills column
    gold_col_candidates = ["gold_skills", "skills", "gold_skill_list"]
    gold_col = next(
        (c for c in gold_col_candidates if c in gold_df.columns), None
    )
    if gold_col is None:
        text_cols = [
            c for c in gold_df.columns
            if gold_df[c].dtype == object and c not in ("course_title",)
        ]
        if not text_cols:
            raise ValueError(
                f"Cannot find a gold-skills column in {gold_csv_path}.\n"
                f"Expected one of {gold_col_candidates} or a text column."
            )
        gold_col = text_cols[0]
        print(f"  Warning: using fallback gold column '{gold_col}'")

    print(f"  Gold skills column  : '{gold_col}'")

    if "course_title" in gold_df.columns:
        df = df.merge(
            gold_df[["course_title", gold_col]].rename(
                columns={gold_col: "gold_skills"}
            ),
            on="course_title",
            how="left",
        )
    else:
        df["gold_skills"] = gold_df[gold_col].values[: len(df)]

    missing = df["gold_skills"].isna().sum()
    if missing:
        print(
            f"  Warning: {missing} course(s) have no gold skills — "
            "they will be skipped in evaluation."
        )
    print(f"  Gold skills loaded  : {df['gold_skills'].notna().sum()}")

    return df


def get_course_context(row: pd.Series) -> Dict[str, str]:
    """Convert a dataset row into a context dict for prompt templates.

    Args:
        row: A single row from the dataset DataFrame.

    Returns:
        Dictionary with keys: title, organization, difficulty, description,
        gold_skills.
    """
    return {
        "title":        str(row.get("course_title", "Unknown")),
        "organization": str(row.get("course_organization", "N/A")),
        "difficulty":   str(row.get("course_difficulty", "Intermediate")),
        "description":  str(row.get("description", "No description available.")),
        "gold_skills":  str(row.get("gold_skills", "")),
    }


DATASET_DF: pd.DataFrame = load_dataset(N_ROWS)
print("Loaded rows:", len(DATASET_DF))


# =============================================================================
# E.  API KEY SETUP
# =============================================================================

def load_groq_api_key(secret_name: str) -> str:
    """Load the Groq API key from Colab Secrets, falling back to getpass.

    Args:
        secret_name: The Colab Secret key name (e.g. "GROQ_ZEROSHOT").

    Returns:
        Groq API key string.

    Raises:
        ValueError: If no key is provided or found.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("API Key Setup")
    print("-" * SEPARATOR_SM)

    api_key = ""

    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get(secret_name).strip()
            print(f"  Loaded key from Colab Secret '{secret_name}'.")
        except Exception:
            print(
                f"  Secret '{secret_name}' not found in Colab Secrets.\n"
                "  Falling back to manual entry."
            )

    if not api_key:
        api_key = getpass.getpass(
            f"  Enter your Groq API key for '{secret_name}': "
        ).strip()

    if not api_key:
        raise ValueError(
            "No Groq API key provided. "
            f"Add it as a Colab Secret named '{secret_name}' "
            "or enter it when prompted."
        )

    print(f"  Key loaded : gsk_...{api_key[-4:]}")
    return api_key


GROQ_API_KEY: str = load_groq_api_key(GROQ_SECRET_NAME)


# =============================================================================
# F.  *** PROMPT ***
#
#  The ctx dict provides:
#    ctx["title"]        — course title
#    ctx["organization"] — course provider
#    ctx["difficulty"]   — Beginner / Intermediate / Advanced
#    ctx["description"]  — cleaned course description
#
#  Output format requirement:
#    The prompt MUST instruct the model to return skills as a
#    comma-separated list on a single line, e.g.:
#    "Python programming, Data analysis, Machine learning"
#    This format is required for the parse_skills() function below.
# =============================================================================


def _build_prompt(ctx: Dict[str, str]) -> str:
    """Return the filled prompt string for this skill extraction experiment.

    IMPORTANT: The output must be a comma-separated list of skills only.
    Do not include numbering, bullets, or explanations in the output.

    Args:
        ctx: Course context dict from get_course_context().

    Returns:
        Fully rendered prompt string ready for the LLM.
    """
    return f"""/no_think
        Extract skills from the following Description.

        Internally:
        Generate multiple independent skill extraction results.
        Compare the outputs.
        Keep only the most consistently identified skills.
        Normalize names and remove duplicates.
        Finalize 2 to 8 skills.

        Important:
        Do not display intermediate outputs or reasoning.
        Return only the final skills as a comma-separated list on a single line.
        Do not include explanations.

        Course Description: {ctx['description']}

        Skills:
        """


print(f"\nPrompt template ready: {TECHNIQUE} / {TASK_TYPE}")


# =============================================================================
# G.  LLM CALLER
# =============================================================================

TPD_ERROR_PHRASES: Tuple[str, ...] = (
    "tokens per day",
    "tpd",
    "on_demand",
    "daily token",
    "token quota",
    "rate limit",
)

RETRY_BACKOFF_MIN: int = 15
RETRY_BACKOFF_MAX: int = 30


class TokenLimitExceeded(Exception):
    """Raised when the Groq daily token quota (TPD) is exhausted.

    No retry is attempted upon catching this exception.
    The caller is responsible for saving any partial results.
    """


def is_token_limit_error(exc: Exception) -> bool:
    """Return True if the exception signals a Groq daily TPD quota error.

    Args:
        exc: Exception raised by the OpenAI client.

    Returns:
        True if any TPD-related phrase is found in the error message.
    """
    return any(phrase in str(exc).lower() for phrase in TPD_ERROR_PHRASES)


def call_llm(
    prompt: str,
    model_name: str,
    max_tokens: int = 1024,
    top_p: float = 0.9,
    retries: int = 3,
) -> Tuple[str, float]:
    """Call the Groq API and return (response_text, latency_seconds).

    Uses the module-level GROQ_API_KEY and TEMPERATURE.
    Strips <think>...</think> blocks from Qwen3 outputs.
    Raises TokenLimitExceeded immediately on daily quota errors.
    Retries up to `retries` times with random back-off on transient errors.

    Args:
        prompt:     Fully rendered user prompt string.
        model_name: Groq model identifier string.
        max_tokens: Maximum tokens in the completion.
        top_p:      Nucleus sampling parameter.
        retries:    Number of retry attempts on transient errors.

    Returns:
        Tuple of (cleaned_response_text, latency_in_seconds).

    Raises:
        TokenLimitExceeded: If a daily TPD quota error is detected.
    """
    client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)
    messages = [
        {"role": "user", "content": prompt},
    ]

    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"      Waiting {delay:.0f}s...", end=" ", flush=True)
    time.sleep(delay)

    for attempt in range(1, retries + 1):
        t_start = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=TEMPERATURE,
                top_p=top_p,
                max_tokens=max_tokens,
            )
            latency = round(time.perf_counter() - t_start, 3)
            raw_text = response.choices[0].message.content or ""
            clean = re.sub(
                r"<think>.*?</think>", "", raw_text, flags=re.DOTALL
            ).strip()
            output_text = clean if clean else raw_text.strip()
            print(f"OK ({latency}s | temp={TEMPERATURE})")
            return output_text, latency

        except Exception as exc:
            latency = round(time.perf_counter() - t_start, 3)
            if is_token_limit_error(exc):
                raise TokenLimitExceeded(str(exc)) from exc
            print(f"\n      Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                print(f"      Retrying in {backoff:.0f}s...")
                time.sleep(backoff)
            else:
                print("      All retries exhausted. Recording empty output.")
                return "", latency


print("LLM caller ready.")


# =============================================================================
# H.  RUN EXPERIMENT
# =============================================================================

PROMPT_SNIPPET_LENGTH: int = 300


def build_result_record(
    run: int,
    model_name: str,
    idx: int,
    ctx: Dict[str, str],
    output: str,
    latency: float,
    prompt: str,
) -> Dict:
    """Assemble a single result record dict for storage in the results list.

    Args:
        run:        Run number (1-indexed).
        model_name: Groq model identifier.
        idx:        Course row index in the dataset.
        ctx:        Course context dictionary.
        output:     LLM response text (raw comma-separated skills).
        latency:    API call latency in seconds.
        prompt:     Full prompt string (truncated for storage).

    Returns:
        Dictionary with all result fields.
    """
    prompt_clean = re.sub(r"\s+", " ", prompt).strip()
    return {
        "run":             run,
        "model":           model_name,
        "task_type":       TASK_TYPE,
        "technique":       TECHNIQUE,
        "temperature":     TEMPERATURE,
        "top_p":           TOP_P,
        "course_idx":      idx,
        "course_title":    ctx["title"],
        "difficulty":      ctx["difficulty"],
        "description_ref": ctx["description"],
        "output":          output,
        "gold_skills":     ctx.get("gold_skills", ""),
        "latency_seconds": latency,
        "prompt_snippet":  prompt_clean[:PROMPT_SNIPPET_LENGTH] + "...",
        "timestamp":       datetime.now(timezone.utc).isoformat(),
    }


def print_token_limit_summary(
    all_records: List[Dict], exc: TokenLimitExceeded
) -> None:
    """Print a structured summary when the daily token limit is hit.

    Args:
        all_records: Records collected before the limit was reached.
        exc:         The TokenLimitExceeded exception that was raised.
    """
    last = all_records[-1] if all_records else {}
    print("\n" + "!" * SEPARATOR_LG)
    print("*** TOKEN LIMIT REACHED — experiment stopped here ***")
    print(f"  Model             : {last.get('model', 'N/A')}")
    print(f"  Task Type         : {last.get('task_type', 'N/A')}")
    print(f"  Technique         : {last.get('technique', 'N/A')}")
    print(f"  Run               : {last.get('run', 'N/A')}")
    print(f"  Course            : {last.get('course_title', 'N/A')}")
    print(f"  Records collected : {len(all_records)}")
    print(f"  Error             : {exc}")
    print("!" * SEPARATOR_LG)


all_records: List[Dict] = []
token_limit_hit: bool = False
experiment_ts: str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\n" + "=" * SEPARATOR_LG)
print("STARTING EXPERIMENT")
print("=" * SEPARATOR_LG)

try:
    for run in range(1, NUM_RUNS + 1):
        print(f"\n{'=' * SEPARATOR_LG}")
        print(f"RUN {run} / {NUM_RUNS}")
        print(f"{'=' * SEPARATOR_LG}")
        print(f"\n  [Task: {TASK_TYPE.upper()}]  temperature={TEMPERATURE}")

        for model_name in MODELS:
            print(f"\n    Model: {model_name}")
            print(f"\n      Technique: {TECHNIQUE}")

            for idx, row in DATASET_DF.iterrows():
                ctx = get_course_context(row)
                prompt = _build_prompt(ctx)
                print(
                    f"      [{idx:02d}] {ctx['title'][:50]}...",
                    end=" ",
                )

                output, latency = call_llm(
                    prompt,
                    model_name,
                    max_tokens=MAX_TOKENS,
                    top_p=TOP_P,
                )

                all_records.append(
                    build_result_record(
                        run, model_name, idx, ctx, output, latency, prompt
                    )
                )

except TokenLimitExceeded as exc:
    token_limit_hit = True
    print_token_limit_summary(all_records, exc)

RESULTS_DF: pd.DataFrame = pd.DataFrame(all_records)
experiment_status = "STOPPED EARLY (token limit)" if token_limit_hit else "COMPLETE"
print(f"\nExperiment {experiment_status}. Total records: {len(RESULTS_DF)}")


# =============================================================================
# I.  SKILL PARSING & EVALUATION METRICS
# =============================================================================

# Composite Score = 0.30(F1) + 0.20(Precision) + 0.20(Recall) + 0.20(BERT-F1) + 0.10(Jaccard)
WEIGHTS_WITH_BERT: Dict[str, float] = {
    "avg_f1":        0.30,
    "avg_precision": 0.20,
    "avg_recall":    0.20,
    "avg_bert_f1":   0.20,
    "avg_jaccard":   0.10,
}

# Weights when BERTScore is NOT available (redistribute BERT weight to F1)
WEIGHTS_WITHOUT_BERT: Dict[str, float] = {
    "avg_f1":        0.50,
    "avg_precision": 0.20,
    "avg_recall":    0.20,
    "avg_jaccard":   0.10,
}

BERTSCORE_BATCH_SIZE: int = 8
BERTSCORE_CANDIDATE_MODELS: List[str] = [
    "roberta-base",
    "xlm-roberta-base",
    "bert-base-uncased",
]


def parse_skills(raw_output: str) -> List[str]:
    """Parse a comma-separated skill list from raw LLM output.

    Handles common formatting artefacts: numbered lists, bullet points,
    extra whitespace, and newlines within the response.

    Args:
        raw_output: Raw text string returned by the LLM.

    Returns:
        List of stripped, non-empty skill strings.
    """
    if not raw_output or not raw_output.strip():
        return []
    # Remove numbered list markers like "1. ", "2) "
    cleaned = re.sub(r"^\d+[.)\-]\s*", "", raw_output, flags=re.MULTILINE)
    # Remove bullet markers
    cleaned = re.sub(r"^[-*•]\s*", "", cleaned, flags=re.MULTILINE)
    # Normalize newlines to commas
    cleaned = re.sub(r"[\n\r]+", ",", cleaned)
    # Split by comma
    skills = [s.strip() for s in cleaned.split(",")]
    # Filter out empty strings and very short tokens
    return [s for s in skills if len(s) > 2]


def normalise_skill(skill: str) -> str:
    """Lowercase and strip punctuation from a skill string for comparison.

    Args:
        skill: A single skill string.

    Returns:
        Normalised lowercase string.
    """
    return re.sub(r"[^a-z0-9 ]", "", skill.lower()).strip()


def skill_set_precision_recall_f1(
    predicted: List[str], gold: List[str]
) -> Tuple[float, float, float]:
    """Compute exact-match set-based precision, recall, and F1 for skills.

    Normalises both lists before comparison.

    Args:
        predicted: List of predicted skill strings.
        gold:      List of gold/reference skill strings.

    Returns:
        Tuple of (precision, recall, f1) each rounded to 4 decimal places.
    """
    pred_set: Set[str] = {normalise_skill(s) for s in predicted if s.strip()}
    gold_set: Set[str] = {normalise_skill(s) for s in gold if s.strip()}

    if not pred_set and not gold_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not gold_set:
        return 0.0, 0.0, 0.0

    tp = len(pred_set & gold_set)
    precision = round(tp / len(pred_set), 4)
    recall = round(tp / len(gold_set), 4)
    f1 = (
        round(2 * precision * recall / (precision + recall), 4)
        if (precision + recall) > 0 else 0.0
    )
    return precision, recall, f1


def jaccard_similarity(predicted: List[str], gold: List[str]) -> float:
    """Compute Jaccard similarity between predicted and gold skill sets.

    Args:
        predicted: List of predicted skill strings.
        gold:      List of gold/reference skill strings.

    Returns:
        Jaccard index in [0, 1] rounded to 4 decimal places.
    """
    pred_set: Set[str] = {normalise_skill(s) for s in predicted if s.strip()}
    gold_set: Set[str] = {normalise_skill(s) for s in gold if s.strip()}
    union = pred_set | gold_set
    if not union:
        return 1.0
    return round(len(pred_set & gold_set) / len(union), 4)


def partial_skill_overlap(predicted: List[str], gold: List[str]) -> float:
    """Compute partial (token-level) overlap between predicted and gold skills.

    For each predicted skill, checks if any of its words appear in any
    gold skill, giving credit for partial matches.

    Args:
        predicted: List of predicted skill strings.
        gold:      List of gold/reference skill strings.

    Returns:
        Mean partial overlap score in [0, 1] rounded to 4 decimal places.
    """
    if not predicted or not gold:
        return 0.0

    gold_words: Set[str] = {
        word
        for s in gold
        for word in normalise_skill(s).split()
        if len(word) > 2
    }

    scores = []
    for skill in predicted:
        skill_words = {
            w for w in normalise_skill(skill).split() if len(w) > 2
        }
        if not skill_words:
            scores.append(0.0)
            continue
        overlap = len(skill_words & gold_words) / len(skill_words)
        scores.append(overlap)

    return round(sum(scores) / len(scores), 4)


def count_accuracy(predicted: List[str], gold: List[str]) -> float:
    """Measure how closely the predicted skill count matches the gold count.

    Returns 1.0 when counts match exactly, decreasing linearly with deviation.

    Args:
        predicted: List of predicted skill strings.
        gold:      List of gold/reference skill strings.

    Returns:
        Count accuracy score in [0, 1] rounded to 4 decimal places.
    """
    if not gold:
        return 1.0 if not predicted else 0.0
    diff = abs(len(predicted) - len(gold))
    return round(max(0.0, 1.0 - diff / len(gold)), 4)


def compute_bertscore(
    hypotheses: List[str], references: List[str]
) -> List[float]:
    """Compute BERTScore F1 for each (hypothesis, reference) skill-list pair.

    Each hypothesis and reference are joined skill strings.
    Tries candidate models in order; returns NaN per item if all fail.

    Args:
        hypotheses: List of predicted skill strings (joined with commas).
        references: List of gold skill strings (joined with commas).

    Returns:
        List of BERTScore F1 values (float), one per pair.
    """
    global BERT_WARNED
    if not BERT_AVAILABLE:
        if not BERT_WARNED:
            print("[BERTScore] Not available — bert_f1 will be NaN.")
            print("  Run Cell 1, restart runtime, then re-run Cell 2.")
            BERT_WARNED = True
        return [float("nan")] * len(hypotheses)

    for model_type in BERTSCORE_CANDIDATE_MODELS:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                _, _, f_scores = bert_score_fn(
                    hypotheses,
                    references,
                    lang="en",
                    model_type=model_type,
                    verbose=False,
                    batch_size=BERTSCORE_BATCH_SIZE,
                )
            print(f"[BERTScore] Model used: {model_type}")
            return [round(float(score), 4) for score in f_scores]
        except Exception as exc:
            print(f"[BERTScore] Failed with {model_type}: {exc}")

    print("[BERTScore] All models failed — bert_f1 = NaN.")
    print("  Workaround: pip install transformers==4.35.2 bert-score==0.3.13")
    return [float("nan")] * len(hypotheses)


def compute_row_metrics(
    row: pd.Series,
    bert_f1_scores: List[float],
    row_index: int,
) -> Dict:
    """Compute all skill extraction evaluation metrics for a single result row.

    Args:
        row:            A row from RESULTS_DF.
        bert_f1_scores: Pre-computed BERTScore F1 values (one per row).
        row_index:      Index of this row in the BERTScore list.

    Returns:
        Dictionary of metric values for this row.
    """
    output = str(row.get("output", "") or "")
    gold_raw = str(row.get("gold_skills", "") or "")

    predicted_skills = parse_skills(output)
    gold_skills = [s.strip() for s in gold_raw.split(",") if s.strip()]

    precision, recall, f1 = skill_set_precision_recall_f1(
        predicted_skills, gold_skills
    )

    return {
        "run":               row["run"],
        "model":             row["model"],
        "temperature":       TEMPERATURE,
        "course_idx":        row["course_idx"],
        "predicted_count":   len(predicted_skills),
        "gold_count":        len(gold_skills),
        "precision":         precision,
        "recall":            recall,
        "f1":                f1,
        "jaccard":           jaccard_similarity(predicted_skills, gold_skills),
        "bert_f1":           bert_f1_scores[row_index],
        "latency_seconds":   row.get("latency_seconds", float("nan")),
    }


def compute_composite_score(
    agg: pd.DataFrame, bert_available: bool
) -> pd.Series:
    """Compute a weighted composite score from aggregated metric columns.

    Args:
        agg:            Aggregated metrics DataFrame.
        bert_available: Whether BERTScore values are available.

    Returns:
        Series of composite scores rounded to 4 decimal places.
    """
    w = WEIGHTS_WITH_BERT if bert_available else WEIGHTS_WITHOUT_BERT
    return (
        agg["avg_f1"]                    * w["avg_f1"]
        + agg["avg_precision"]           * w["avg_precision"]
        + agg["avg_recall"]              * w["avg_recall"]
        + agg["avg_bert_f1"].fillna(0)   * w.get("avg_bert_f1", 0)
        + agg["avg_jaccard"]             * w["avg_jaccard"]
    ).round(4)


def evaluate(results_df: pd.DataFrame) -> pd.DataFrame:
    """Compute and aggregate all evaluation metrics from the results DataFrame.

    Groups by (run, model) and computes mean values for each metric
    plus a weighted composite score.

    Args:
        results_df: Raw results DataFrame from the experiment loop.

    Returns:
        Aggregated metrics DataFrame sorted by composite_score descending.
    """
    if results_df.empty:
        print("No results to evaluate.")
        return pd.DataFrame()

    print("\nParsing predicted skills and computing BERTScore...")
    print("(downloads roberta-base ~500 MB on first run)")

    # Join skill lists as strings for BERTScore
    hyp_strings = [
        ", ".join(parse_skills(str(out)))
        for out in results_df["output"].fillna("")
    ]
    ref_strings = results_df["gold_skills"].fillna("").tolist()

    bert_scores = compute_bertscore(hyp_strings, ref_strings)

    print("Computing row-level metrics...")
    row_metric_records: List[Dict] = [
        compute_row_metrics(row, bert_scores, i)
        for i, (_, row) in enumerate(results_df.iterrows())
    ]
    per_row_df = pd.DataFrame(row_metric_records)

    print("Aggregating...")
    agg = per_row_df.groupby(["run", "model"]).agg(
        avg_precision=      ("precision",      "mean"),
        avg_recall=         ("recall",         "mean"),
        avg_f1=             ("f1",             "mean"),
        avg_jaccard=        ("jaccard",        "mean"),
        avg_bert_f1=        ("bert_f1",        "mean"),
        avg_predicted_count=("predicted_count","mean"),
        avg_gold_count=     ("gold_count",     "mean"),
        avg_latency_sec=    ("latency_seconds","mean"),
        min_latency_sec=    ("latency_seconds","min"),
        max_latency_sec=    ("latency_seconds","max"),
        total_latency_sec=  ("latency_seconds","sum"),
        total_samples=      ("course_idx",     "count"),
    ).round(4).reset_index()

    agg.insert(0, "technique",    TECHNIQUE)
    agg.insert(1, "task_type",    TASK_TYPE)
    agg.insert(2, "temperature",  TEMPERATURE)

    bert_is_available = agg["avg_bert_f1"].notna().any()
    agg["composite_score"] = compute_composite_score(agg, bert_is_available)

    return agg.sort_values(
        "composite_score", ascending=False
    ).reset_index(drop=True)


METRICS_DF: pd.DataFrame = evaluate(RESULTS_DF)
print(f"Metrics computed: {len(METRICS_DF)} rows")


# =============================================================================
# J.  FULL METRICS TABLE
# =============================================================================

DISPLAY_COLUMNS: List[str] = [
    "run", "model", "task_type", "technique", "temperature",
    "avg_precision", "avg_recall", "avg_f1", "avg_jaccard",
    "avg_bert_f1",
    "avg_predicted_count", "avg_gold_count",
    "avg_latency_sec", "composite_score",
]


def print_full_metrics_table(metrics_df: pd.DataFrame) -> None:
    """Configure pandas display and print the full metrics table.

    Args:
        metrics_df: Aggregated metrics DataFrame.
    """
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 240)
    pd.set_option("display.float_format", "{:.4f}".format)

    visible_cols = [
        col for col in DISPLAY_COLUMNS if col in metrics_df.columns
    ]
    print("\n" + "=" * SEPARATOR_XL)
    print("FULL METRICS TABLE — SKILL EXTRACTION")
    print("=" * SEPARATOR_XL)
    print(metrics_df[visible_cols].to_string(index=False))


print_full_metrics_table(METRICS_DF)


# =============================================================================
# K.  FINAL SUMMARY — BEST & WORST MODEL
# =============================================================================

SUMMARY_METRIC_COLS: List[str] = [
    "avg_precision", "avg_recall", "avg_f1", "avg_jaccard",
    "avg_bert_f1",
    "avg_predicted_count", "avg_gold_count",
    "avg_latency_sec", "composite_score",
]


def build_best_worst_summary(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Build a DataFrame ranking best and worst models for this experiment.

    Averages composite_score across all runs before ranking.

    Args:
        metrics_df: Aggregated metrics DataFrame from evaluate().

    Returns:
        DataFrame with one BEST row and one WORST row.
    """
    if metrics_df.empty:
        return pd.DataFrame()

    valid_cols = [
        col for col in SUMMARY_METRIC_COLS if col in metrics_df.columns
    ]
    grouped = (
        metrics_df
        .groupby("model")[valid_cols]
        .mean()
        .round(4)
        .reset_index()
        .sort_values("composite_score", ascending=False)
        .reset_index(drop=True)
    )

    best = grouped.iloc[0].to_dict()
    worst = grouped.iloc[-1].to_dict()
    best["rank"] = "BEST"
    worst["rank"] = "WORST"

    summary = pd.DataFrame([best, worst])
    summary.insert(0, "technique",   TECHNIQUE)
    summary.insert(1, "task_type",   TASK_TYPE)
    summary.insert(2, "temperature", TEMPERATURE)
    return summary


def format_bert_value(row: pd.Series) -> str:
    """Format the avg_bert_f1 value for display, returning N/A if unavailable.

    Args:
        row: A row from the summary DataFrame.

    Returns:
        Formatted string or '  N/A  ' if NaN.
    """
    if "avg_bert_f1" in row.index and not math.isnan(row["avg_bert_f1"]):
        return f"{row['avg_bert_f1']:.4f}"
    return "  N/A  "


def print_best_worst_summary(summary_df: pd.DataFrame) -> None:
    """Print a formatted best-and-worst model summary table to stdout.

    Args:
        summary_df: DataFrame produced by build_best_worst_summary().
    """
    if summary_df.empty:
        print("No data for best/worst summary.")
        return

    print("\n" + "=" * SEPARATOR_XXL)
    print("FINAL SUMMARY — BEST & WORST MODEL (SKILL EXTRACTION)")
    print(f"Technique: {TECHNIQUE}  |  Task: {TASK_TYPE}  |  Temperature: {TEMPERATURE}")
    print("(composite_score averaged across all runs; higher = better)")
    print("=" * SEPARATOR_XXL)
    print(
        f"  {'Rank':<6} {'Model':<32} {'Composite':>9} {'Precision':>10} "
        f"{'Recall':>7} {'F1':>7} {'Jaccard':>8} "
        f"{'BERT_F1':>8} {'Latency':>8}"
    )
    print("  " + "-" * TABLE_ROW_WIDTH)

    for _, row in summary_df.iterrows():
        bert_str = format_bert_value(row)
        rank_marker = ">>>" if row["rank"] == "BEST" else "   "
        print(
            f"  {rank_marker} {row['rank']:<3} "
            f"{row['model']:<32} "
            f"{row['composite_score']:>9.4f} "
            f"{row['avg_precision']:>10.4f} "
            f"{row['avg_recall']:>7.4f} "
            f"{row['avg_f1']:>7.4f} "
            f"{row['avg_jaccard']:>8.4f} "
            f"{bert_str:>8} "
            f"{row['avg_latency_sec']:>8.3f}s"
        )

    print("\n" + "=" * SEPARATOR_XXL)
    print("KEY")
    print("  >>> BEST      = highest composite_score across all models")
    print("      WORST     = lowest composite_score across all models")
    print("  Composite     = 0.30×F1 + 0.20×Precision + 0.20×Recall + 0.20×BERT-F1 + 0.10×Jaccard (higher = better)")
    print("  Precision     = fraction of predicted skills that are in gold set")
    print("  Recall        = fraction of gold skills that were predicted")
    print("  F1            = harmonic mean of Precision and Recall (weight: 0.30)")
    print("  Jaccard       = intersection / union of predicted and gold skill sets (weight: 0.10)")
    print("  BERT_F1       = semantic similarity via BERTScore (roberta-base) (weight: 0.20)")
    print("  Composite     = 0.30×F1 + 0.20×Precision + 0.20×Recall + 0.20×BERT-F1 + 0.10×Jaccard")
    print("=" * SEPARATOR_XXL)


SUMMARY_DF: pd.DataFrame = build_best_worst_summary(METRICS_DF)
print_best_worst_summary(SUMMARY_DF)


# =============================================================================
# L.  SAVE & DOWNLOAD
# =============================================================================

_file_tag: str = f"{TECHNIQUE}_{TASK_TYPE}_{experiment_ts}"
RESULTS_FILENAME: str = f"skills_results_{_file_tag}.csv"
METRICS_FILENAME: str = f"skills_metrics_{_file_tag}.csv"
SUMMARY_FILENAME: str = f"skills_best_worst_{_file_tag}.csv"


def save_results(
    results_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    in_colab: bool,
) -> None:
    """Save all result DataFrames to CSV and optionally trigger downloads.

    Args:
        results_df: Raw results from the experiment loop.
        metrics_df: Aggregated metrics DataFrame.
        summary_df: Best/worst summary DataFrame.
        in_colab:   True if running inside Google Colab.
    """
    results_df.to_csv(RESULTS_FILENAME, index=False)
    metrics_df.to_csv(METRICS_FILENAME, index=False)
    if not summary_df.empty:
        summary_df.to_csv(SUMMARY_FILENAME, index=False)

    print(f"\nSaved: {RESULTS_FILENAME}  ({len(results_df)} rows)")
    print(f"Saved: {METRICS_FILENAME}  ({len(metrics_df)} rows)")
    print(f"Saved: {SUMMARY_FILENAME}  ({len(summary_df)} rows)")

    if in_colab:
        print("\nDownloading files...")
        files.download(RESULTS_FILENAME)
        files.download(METRICS_FILENAME)
        files.download(SUMMARY_FILENAME)
        print("Downloads initiated.")
    else:
        print("\nFiles saved locally.")

    print("\nDone.")


save_results(RESULTS_DF, METRICS_DF, SUMMARY_DF, IN_COLAB)

LIBRARY VERSIONS (active runtime)
  openai             2.32.0
  pandas             2.2.2
  numpy              2.0.2
  torch              2.10.0+cpu
  transformers       4.35.2
  bert_score         0.3.12
  matplotlib         3.10.0
  scipy              1.16.3
  BERTScore available : True
Experiment Configuration
----------------------------------------
  Technique          : self_consistency
  Task type          : skill_extraction
  Temperature        : 0.2
  Runs               : 1
  Models             : ['llama-3.1-8b-instant', 'qwen/qwen3-32b', 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'llama-3.3-70b-versatile']
  Courses            : 25
  Total API calls    : 125
  Est. delay time    : ~19.8 min
  top_p              : 0.9

----------------------------------------
Dataset Load
----------------------------------------
  Loading Coursera dataset from Google Drive...
  Please wait...

Mounted at /content/drive
  Found dataset at: /content/drive/MyDrive/research/data/coursera_clean_sa

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads initiated.

Done.
